# 04 Business Interpretation

Phase 4A translates the completed PD and ECL outputs into business findings, dashboard-ready summary tables, resume material, and interview talking points. This notebook does not retrain the PD model, change ECL assumptions, or build the Streamlit dashboard.

## Project Context

This portfolio project demonstrates a full starter workflow for credit risk analytics: data understanding, baseline PD modeling, simplified IFRS 9-style ECL calculation, and business interpretation.

## Business Objective

Convert row-level ECL results into concise portfolio insights that can support an interview discussion, a GitHub project README, and a future dashboard.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
PREDICTIONS_DIR = OUTPUTS_DIR / "predictions"
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

## Load ECL Outputs

In [ ]:
ecl_path = OUTPUTS_DIR / "ecl_results.csv"
scenario_path = PREDICTIONS_DIR / "ecl_scenario_summary.csv"

ecl = pd.read_csv(ecl_path, low_memory=False)
scenarios = pd.read_csv(scenario_path)

print(f"Loaded ECL results: {ecl_path}")
print(f"Loaded scenario summary: {scenario_path}")
print(f"ECL shape: {ecl.shape}")
display(ecl.head())
display(scenarios)

## Portfolio Overview

In [ ]:
total_loans = len(ecl)
total_exposure = ecl["ead"].sum()
total_ecl = ecl["ecl"].sum()
ecl_rate = total_ecl / total_exposure
average_pd = ecl["pd_score"].mean()
average_lgd = ecl["lgd"].mean()

overview = pd.DataFrame([
    {
        "total_loans": total_loans,
        "total_exposure": total_exposure,
        "total_ecl": total_ecl,
        "ecl_rate": ecl_rate,
        "average_pd": average_pd,
        "average_lgd": average_lgd,
    }
])
display(overview)

## PD Model Summary

The PD input is a baseline logistic regression model from Phase 2. It is a benchmark model, not a production credit decision model. Phase 2 test metrics were ROC AUC 0.701, accuracy 0.632, precision 0.285, recall 0.651, and F1 0.396.

## ECL Methodology Summary

- EAD uses `loan_amnt`.
- LGD uses a simplified home-ownership rule.
- ECL is calculated as `pd_score x lgd x ead`.
- Stress scenarios apply PD and LGD multipliers, capped at 100%.

## IFRS 9-style Staging Interpretation

In [ ]:
def summarize_group(df, group_col):
    summary = (
        df.groupby(group_col, dropna=False, observed=False)
        .agg(
            loan_count=("ecl", "size"),
            total_exposure=("ead", "sum"),
            total_ecl=("ecl", "sum"),
            average_pd=("pd_score", "mean"),
            average_lgd=("lgd", "mean"),
        )
        .reset_index()
        .rename(columns={group_col: "group_value"})
    )
    summary["ecl_rate"] = summary["total_ecl"] / summary["total_exposure"]
    return summary[["group_value", "loan_count", "total_exposure", "total_ecl", "ecl_rate", "average_pd", "average_lgd"]].sort_values("total_ecl", ascending=False)

stage_summary = summarize_group(ecl, "ifrs9_stage")
display(stage_summary)

## Risk Concentration Analysis

In [ ]:
score_band_summary = summarize_group(ecl, "pd_score_band")
display(score_band_summary)

grade_summary = summarize_group(ecl, "grade") if "grade" in ecl.columns else pd.DataFrame()
purpose_summary = summarize_group(ecl, "purpose") if "purpose" in ecl.columns else pd.DataFrame()

if not grade_summary.empty:
    display(grade_summary)
if not purpose_summary.empty:
    display(purpose_summary)

## Scenario Analysis Interpretation

In [ ]:
scenario_view = scenarios.copy()
scenario_view["ecl_increase_vs_base"] = scenario_view["total_ecl"] - scenario_view.loc[scenario_view["scenario"] == "Base", "total_ecl"].iloc[0]
scenario_view["ecl_increase_pct_vs_base"] = scenario_view["ecl_increase_vs_base"] / scenario_view.loc[scenario_view["scenario"] == "Base", "total_ecl"].iloc[0]
display(scenario_view)

## Business Recommendations

- Use Stage 3 and high score band views as first dashboard filters.
- Keep the base, mild stress, and severe stress scenario comparison prominent.
- Present ECL concentration by grade and purpose as business-facing portfolio diagnostics.
- Keep the limitations visible because this is a simplified analytics prototype.

## Company-specific Relevance

The project is most relevant for roles involving credit risk, portfolio analytics, risk reporting, data analytics, fintech lending, financial research, and model documentation.

## Resume Bullet Extraction

The strongest resume angle is that the project connects Python, credit risk modeling, transparent ECL assumptions, scenario analysis, and business-ready reporting.

## Interview Talking Points

- Why logistic regression was used as the baseline.
- How PD, LGD, and EAD connect to ECL.
- Why the staging logic is simplified.
- How model limitations were documented instead of hidden.
- How the outputs can feed a dashboard.

## Limitations

- This project uses a public LendingClub-style dataset, not bank production data.
- PD scores are from a baseline model and are not calibrated regulatory PDs.
- EAD, LGD, staging, and scenario assumptions are simplified.
- The Streamlit dashboard has not been built yet.

## Next Steps for Dashboard

In [ ]:
stage_lookup = stage_summary.set_index("group_value")
dashboard_summary = pd.DataFrame([
    {
        "total_loans": total_loans,
        "total_exposure": total_exposure,
        "total_ecl": total_ecl,
        "ecl_rate": ecl_rate,
        "average_pd": average_pd,
        "average_lgd": average_lgd,
        "stage_1_count": int(stage_lookup.loc["Stage 1", "loan_count"]) if "Stage 1" in stage_lookup.index else 0,
        "stage_2_count": int(stage_lookup.loc["Stage 2", "loan_count"]) if "Stage 2" in stage_lookup.index else 0,
        "stage_3_count": int(stage_lookup.loc["Stage 3", "loan_count"]) if "Stage 3" in stage_lookup.index else 0,
        "stage_1_ecl": float(stage_lookup.loc["Stage 1", "total_ecl"]) if "Stage 1" in stage_lookup.index else 0.0,
        "stage_2_ecl": float(stage_lookup.loc["Stage 2", "total_ecl"]) if "Stage 2" in stage_lookup.index else 0.0,
        "stage_3_ecl": float(stage_lookup.loc["Stage 3", "total_ecl"]) if "Stage 3" in stage_lookup.index else 0.0,
    }
])

dashboard_summary.to_csv(PREDICTIONS_DIR / "dashboard_summary.csv", index=False)
stage_summary.to_csv(PREDICTIONS_DIR / "ecl_by_stage.csv", index=False)
score_band_summary.to_csv(PREDICTIONS_DIR / "ecl_by_score_band.csv", index=False)
if not grade_summary.empty:
    grade_summary.to_csv(PREDICTIONS_DIR / "ecl_by_grade.csv", index=False)
if not purpose_summary.empty:
    purpose_summary.to_csv(PREDICTIONS_DIR / "ecl_by_purpose.csv", index=False)

display(dashboard_summary)
print("Saved dashboard-ready CSV tables to outputs/predictions/.")